# Employee Hierarchy Validation

This notebook validates the improved organizational hierarchy for the 50-employee synthetic workforce sample.

The hierarchy contains:

- Department heads
- Team managers
- Individual contributors

The main rules are:

- Every department has one department head.
- Department heads have no manager in the current simplified model.
- Team managers report to department heads.
- Individual contributors have valid managers.
- Employees and their managers belong to the same department.
- Referenced managers are active.
- No manager has more than six direct reports.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

employees = pd.read_csv(
    RAW_DATA_DIR / "employees_sample.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

departments = pd.read_csv(
    RAW_DATA_DIR / "departments.csv"
)

employees["manager_id"] = (
    employees["manager_id"]
    .astype("Int64")
)

print(employees.shape)

(50, 15)


## 1. Organizational-level counts

In [2]:
organizational_level_counts = (
    employees["organizational_level"]
    .value_counts()
    .rename_axis("organizational_level")
    .reset_index(name="employee_count")
)

organizational_level_counts

,organizational_level,employee_count
0,Individual Contributor,38
1,Department Head,8
2,Team Manager,4


In [3]:
manager_lookup = (
    employees
    .set_index("employee_id")
    [
        [
            "first_name",
            "last_name",
            "department_id",
            "employment_status",
            "organizational_level",
        ]
    ]
)

manager_lookup.head()

,first_name,last_name,department_id,employment_status,organizational_level
employee_id,,,,,
10001,Danielle,Johnson,1,Active,Department Head
10002,Joshua,Walker,1,Active,Team Manager
10003,Jill,Rhodes,1,Active,Team Manager
10004,Patricia,Miller,1,Active,Individual Contributor
10005,Robert,Johnson,1,Active,Individual Contributor


## 2. Department-head checks

In [4]:
department_heads = employees[
    employees["organizational_level"]
    == "Department Head"
]

department_head_checks = pd.Series(
    {
        "there are eight department heads": (
            len(department_heads) == 8
        ),
        "every department has one head": (
            department_heads[
                "department_id"
            ].nunique()
            == len(departments)
        ),
        "department heads have no manager": (
            department_heads[
                "manager_id"
            ].isna().all()
        ),
        "all department heads are active": (
            department_heads[
                "employment_status"
            ].eq("Active").all()
        ),
    },
    name="passed",
)

department_head_checks

there are eight department heads    True
every department has one head       True
department heads have no manager    True
all department heads are active     True
Name: passed, dtype: bool

## 3. Team-manager checks

In [5]:
team_managers = employees[
    employees["organizational_level"]
    == "Team Manager"
].copy()

team_managers["manager_level"] = (
    team_managers["manager_id"]
    .astype(int)
    .map(
        manager_lookup[
            "organizational_level"
        ]
    )
)

team_manager_checks = pd.Series(
    {
        "team managers have manager IDs": (
            team_managers[
                "manager_id"
            ].notna().all()
        ),
        "team managers report to department heads": (
            team_managers[
                "manager_level"
            ].eq(
                "Department Head"
            ).all()
        ),
        "all team managers are active": (
            team_managers[
                "employment_status"
            ].eq("Active").all()
        ),
    },
    name="passed",
)

team_manager_checks

team managers have manager IDs              True
team managers report to department heads    True
all team managers are active                True
Name: passed, dtype: bool

## 4. Individual-contributor checks

In [6]:
individual_contributors = employees[
    employees["organizational_level"]
    == "Individual Contributor"
].copy()

individual_contributors[
    "manager_level"
] = (
    individual_contributors[
        "manager_id"
    ]
    .astype(int)
    .map(
        manager_lookup[
            "organizational_level"
        ]
    )
)

individual_contributor_checks = (
    pd.Series(
        {
            "individual contributors have managers": (
                individual_contributors[
                    "manager_id"
                ].notna().all()
            ),
            "managers have valid levels": (
                individual_contributors[
                    "manager_level"
                ].isin(
                    [
                        "Department Head",
                        "Team Manager",
                    ]
                ).all()
            ),
        },
        name="passed",
    )
)

individual_contributor_checks

individual contributors have managers    True
managers have valid levels               True
Name: passed, dtype: bool

## 5. Employee-manager department checks

In [7]:
managed_employees = employees[
    employees["manager_id"].notna()
].copy()

managed_employees[
    "manager_department_id"
] = (
    managed_employees["manager_id"]
    .astype(int)
    .map(
        manager_lookup[
            "department_id"
        ]
    )
)

managed_employees[
    "same_department_as_manager"
] = (
    managed_employees["department_id"]
    == managed_employees[
        "manager_department_id"
    ]
)

managed_employees[
    "same_department_as_manager"
].value_counts()

same_department_as_manager
True    42
Name: count, dtype: int64

In [8]:
managed_employees[
    "manager_status"
] = (
    managed_employees["manager_id"]
    .astype(int)
    .map(
        manager_lookup[
            "employment_status"
        ]
    )
)

managed_employees[
    "manager_status"
].value_counts()

manager_status
Active    42
Name: count, dtype: int64

## 6. Span-of-control analysis

In [9]:
direct_report_counts = (
    employees["manager_id"]
    .dropna()
    .astype(int)
    .value_counts()
    .rename_axis("manager_id")
    .reset_index(name="direct_reports")
)

direct_report_counts

,manager_id,direct_reports
0,10012,5
1,10023,5
2,10002,4
3,10013,4
4,10029,4
5,10042,4
6,10003,3
7,10034,3
8,10038,3
9,10047,3


In [10]:
manager_summary = (
    direct_report_counts
    .merge(
        employees[
            [
                "employee_id",
                "first_name",
                "last_name",
                "department_id",
                "organizational_level",
            ]
        ],
        left_on="manager_id",
        right_on="employee_id",
        how="left",
    )
    .merge(
        departments[
            [
                "department_id",
                "department_name",
            ]
        ],
        on="department_id",
        how="left",
    )
)

manager_summary[
    [
        "manager_id",
        "first_name",
        "last_name",
        "department_name",
        "organizational_level",
        "direct_reports",
    ]
].sort_values(
    "direct_reports",
    ascending=False,
)

,manager_id,first_name,last_name,department_name,organizational_level,direct_reports
0,10012,Matthew,Moore,Manufacturing,Team Manager,5
1,10023,Christopher,Hall,Supply Chain,Department Head,5
2,10002,Joshua,Walker,Engineering,Team Manager,4
3,10013,Susan,Rogers,Manufacturing,Team Manager,4
4,10029,George,Daniel,Sales,Department Head,4
5,10042,Brenda,Hurst,Information Technology,Department Head,4
6,10003,Jill,Rhodes,Engineering,Team Manager,3
7,10034,Jeremy,Johnson,Finance,Department Head,3
8,10038,Diana,Foster,Human Resources,Department Head,3
9,10047,Laura,Henderson,Customer Support,Department Head,3


## 7. Final hierarchy validation

In [11]:
hierarchy_checks = pd.concat(
    [
        department_head_checks,
        team_manager_checks,
        individual_contributor_checks,
    ]
)

additional_checks = pd.Series(
    {
        "employees and managers share departments": (
            managed_employees[
                "same_department_as_manager"
            ].all()
        ),
        "all referenced managers are active": (
            managed_employees[
                "manager_status"
            ].eq("Active").all()
        ),
        "no manager exceeds six direct reports": (
            manager_summary[
                "direct_reports"
            ].le(6).all()
        ),
    },
    name="passed",
)

all_hierarchy_checks = pd.concat(
    [
        hierarchy_checks,
        additional_checks,
    ]
)

hierarchy_validation = pd.DataFrame(
    {
        "check": all_hierarchy_checks.index,
        "passed": all_hierarchy_checks.values,
    }
)

hierarchy_validation

,check,passed
0,there are eight department heads,True
1,every department has one head,True
2,department heads have no manager,True
3,all department heads are active,True
4,team managers have manager IDs,True
5,team managers report to department heads,True
6,all team managers are active,True
7,individual contributors have managers,True
8,managers have valid levels,True
9,employees and managers share departments,True


In [12]:
if hierarchy_validation["passed"].all():
    print(
        "All hierarchy validation checks passed."
    )
else:
    print(
        "One or more hierarchy checks failed."
    )

All hierarchy validation checks passed.


## 8. Conclusions

The improved employee generator creates a more realistic organizational hierarchy.

### Successful checks

- Every department has exactly one department head.
- Department heads are active and have no manager in the simplified model.
- Larger departments contain team managers.
- Team managers report to department heads.
- Individual contributors report to either department heads or team managers.
- Employees and their managers belong to the same department.
- Every referenced manager is active.
- No manager has more than six direct reports.

### Remaining limitations

- Department heads do not yet report to an executive.
- Managers currently use an existing departmental job role rather than a dedicated management title.
- Historical manager changes are not yet modeled.
- Terminated employees retain their most recent manager ID.
- The current dataset still contains only 50 employees.

These limitations can be addressed in later versions.